# 07. Gradient updates and parameter schedules

![Schedule and provenance](../images/07_gradient_updates_and_schedules.svg)

This notebook shows how a fixed-update training schedule becomes a scientific contract for the iso-catalog phase-allocation study. Read the [lecture](../lectures/07_gradient_updates_and_schedules.md) and return to the [tutorial index](../README.md).

**Learning goals:** turn clips, batch size, and accumulation into exact updates; verify a shared exposure tier; choose a throughput tier outcome-blind; and store enough provenance to reject an incompatible resume.

In [ ]:
from dataclasses import dataclass
import math
import numpy as np

SEED = 17
rng = np.random.default_rng(SEED)
assert rng.integers(1, 10) >= 1


## Fixed clips imply fixed updates

A comparison is only about allocation when every model receives the same clip exposure and the same effective batch size.

In [ ]:
def completed_updates(clips, per_device_batch, devices, accumulation):
    effective_batch = per_device_batch * devices * accumulation
    if clips % effective_batch:
        raise ValueError('clip exposure must divide the effective batch')
    return clips // effective_batch, effective_batch

for clips in (4_096_000, 8_192_000):
    updates, batch = completed_updates(clips, 32, 8, 2)
    print(clips, updates, batch)
    assert updates * batch == clips


## Choose the tier before outcomes

A storage probe may choose the full or half exposure tier, but it cannot choose a different tier for a favorable allocation.

In [ ]:
def storage_stable(rates):
    rates = np.asarray(rates, dtype=float)
    if rates.shape != (8,):
        raise ValueError('rates.shape != (8,)')
    return rates.min() >= 30.0 and rates.max() / rates.min() <= 1.5

rates = np.array([62, 60, 61, 59, 63, 60, 62, 61], dtype=float)
assert storage_stable(rates)
EXPOSURE = 8_192_000 if storage_stable(rates) else 4_096_000
assert EXPOSURE in {4_096_000, 8_192_000}


## Resume is a protocol check

A checkpoint cannot be resumed just because its model weights load. It must belong to the same allocation, phase catalog, exposure, and random streams.

In [ ]:
REQUIRED_RESUME_FIELDS = {
    'manifest_digest', 'phase_catalog_digest', 'allocation',
    'unique_sequences', 'origins_per_sequence', 'nominal_catalog_size',
    'origin_policy', 'planned_exposure', 'effective_batch',
    'completed_updates', 'optimization_seed', 'replicate_seed',
    'sequence_stream_version', 'phase_stream_version',
    'spatial_stream_version', 'mask_stream_version',
}
ALLOCATIONS = {'breadth', 'balanced', 'phase_depth', 'nearby_jitter'}
ORIGIN_POLICIES = {'base_phase', 'phase_separated', 'nearby_jitter'}

row = {
    'manifest_digest': 'synthetic-manifest-v2',
    'phase_catalog_digest': 'synthetic-phase-v1',
    'allocation': 'phase_depth', 'unique_sequences': 62_500,
    'origins_per_sequence': 4, 'nominal_catalog_size': 250_000,
    'origin_policy': 'phase_separated', 'planned_exposure': EXPOSURE,
    'effective_batch': 512, 'completed_updates': EXPOSURE // 512,
    'optimization_seed': 13, 'replicate_seed': 17,
    'sequence_stream_version': 'sequence-v2',
    'phase_stream_version': 'phase-v1',
    'spatial_stream_version': 'spatial-v1', 'mask_stream_version': 'mask-v1',
}

def validate_resume(saved, expected):
    if set(saved) != REQUIRED_RESUME_FIELDS:
        raise ValueError('resume metadata has missing or unknown fields')
    if saved['allocation'] not in ALLOCATIONS or saved['origin_policy'] not in ORIGIN_POLICIES:
        raise ValueError('unknown allocation or origin policy')
    if saved['nominal_catalog_size'] != saved['unique_sequences'] * saved['origins_per_sequence']:
        raise ValueError('nominal catalog cardinality is inconsistent')
    if saved != expected:
        raise ValueError('resume metadata does not match the frozen row')

validate_resume(row, dict(row))
try:
    changed = dict(row); changed['phase_catalog_digest'] = 'other'
    validate_resume(changed, row)
except ValueError:
    pass
else:
    raise AssertionError('phase-catalog mismatch must fail closed')


**Takeaway:** fixed exposure is necessary but not sufficient. Scientific comparability also requires a fixed allocation definition, phase catalog, stream versions, checkpoint step, and fail-closed resume behavior.

Previous: [06. Representation collapse](06_representation_collapse.ipynb) · Next: [08. Group-aware sampling](08_group_aware_sampling.ipynb)